# Día 6 — Visualización, optimización y proyecto

Conectar PCA con evaluación honesta y búsqueda de hiperparámetros.

## Objetivos de aprendizaje

- Comunicar un conjunto multidimensional mediante PCA.
- Construir baseline y optimizar con validación cruzada agrupada.
- Evitar mezclar países entre entrenamiento y prueba.
- Elegir un dataset pertinente y preparar un proyecto reproducible.

> **Ruta presencial (3 horas):** puente PCA 0–20 min; generalización a países nuevos 20–75; baseline/GroupKFold/búsqueda 75–135; interpretación y taller del proyecto 135–180. El proyecto ya fue planteado en el Día 4.

**Dataset:** indicadores país-año del Banco Mundial descargados localmente en `datos/world_bank_latam_ml.csv`. La celda de búsqueda no accede a servicios externos.

## ¿Qué significa «optimizar» un modelo?

En clase no vamos a forzar que un modelo tenga la métrica más grande a cualquier costo. Definimos una tarea, una métrica, una referencia y un procedimiento de validación; después comparamos configuraciones razonables. La búsqueda de hiperparámetros automatiza comparaciones, pero también puede sobreajustarse a Validation si se prueban demasiadas opciones.

Esta jornada une dos habilidades: representar patrones multidimensionales para explicarlos y seleccionar complejidad predictiva con separación por país. En el contexto del BCE, una mejora numérica pequeña debe traducirse a una utilidad analítica creíble.

### Parámetros, hiperparámetros y score

El modelo aprende **parámetros** de Train (por ejemplo, divisiones y valores de hojas de los árboles). El analista fija **hiperparámetros** antes o durante una búsqueda: `max_depth`, `min_samples_leaf`, cantidad de árboles. La métrica de validación resume rendimiento para comparar alternativas; no modifica por sí sola el objetivo ni define el costo del error.

## 1. Visualización multidimensional como puente a PCA

Retomamos los indicadores de los días 4–5. Un gráfico bidimensional no puede mostrar seis dimensiones originales. PCA proyecta la variación sobre dos ejes; usaremos el color para inspeccionar crecimiento del PIB, no como prueba supervisada ni como predictor causal.

> **Nota para quien viene de estadística:** esta función `buscar_datos()` y su `raise FileNotFoundError(...)` se explican con un ejemplo mínimo en el notebook complementario [Día 0.1, Sección 9.2](00_1_Dia0_Puente_a_Scikit_Learn.ipynb).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

macro = pd.read_csv(DATA_DIR / "world_bank_latam_ml.csv")
features = ["inflacion_pct", "desempleo_pct", "exportaciones_pct_pib",
            "inversion_pct_pib", "crecimiento_poblacion_pct"]
target = "crecimiento_pib_pct"
df = macro.dropna(subset=features+[target]).copy()
Xz = StandardScaler().fit_transform(df[features])
coords = PCA(n_components=2, random_state=42).fit_transform(Xz)
plt.figure(figsize=(8,5))
sc = plt.scatter(coords[:,0], coords[:,1], c=df[target], cmap="coolwarm", alpha=.8)
plt.colorbar(sc,label="Crecimiento PIB (%)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Vista PCA de indicadores macroeconómicos")


### Leer una proyección PCA antes de ajustar modelos

El color del gráfico es crecimiento observado; los ejes PC1 y PC2 son combinaciones calculadas de features estandarizadas. PCA no usa el target para elegir los ejes en este ejemplo. Para interpretar ejes revisa los loadings del Día 5; para comparar países en una dimensión del tiempo, observa fechas y no atribuyas cada grupo a un factor causal aislado.

### Interpreta visualización

Describe agrupaciones, años extremos y solapamiento; revisa loadings de Día 5 para dar significado a los ejes. PCA se ajustó aquí a toda la tabla para una visualización exploratoria; esa figura **no** se reutiliza para estimar desempeño predictivo. Para modelar, cada transformación se ajustará dentro de los folds.

### Qué pierde una vista 2D

Proyectar cinco predictores en dos dimensiones descarta tres direcciones. Observaciones superpuestas en el plano pueden diferir en las dimensiones eliminadas. Separación visual tampoco garantiza que un Random Forest prediga mejor. Usamos esta figura como **visualización exploratoria**, no como evidencia de rendimiento; el modelo predictivo tendrá su propio flujo y validación.

## 2. El problema de optimización

Un hiperparámetro (`max_depth`, `min_samples_leaf`) se fija antes de ajustar el modelo; los parámetros se aprenden de los datos. Probar muchas configuraciones sobre el mismo Validation puede sobreajustar también la selección. Usaremos una validación cruzada interna y compararemos con una referencia sencilla.

## Separar países: ¿qué pregunta estamos respondiendo?

Una división aleatoria por filas puede colocar Brasil 2010 en Train y Brasil 2011 en Test; el modelo ya vio una observación casi vecina del mismo país. Reservar países completos hace una pregunta más exigente: ¿generaliza a economías del panel no vistas durante el ajuste?

Esto no es la única división posible. Para pronosticar años futuros usaríamos un corte temporal; para estimar otro año en los mismos países aplicaríamos validación temporal por país. El split depende del uso pretendido.

### Un split responde a una pregunta concreta

Hay al menos tres preguntas: predecir otro año para países conocidos; predecir un país nunca observado; o pronosticar el futuro para países conocidos. Cada una pide una partición distinta. Reservar países completos responde la segunda, aunque siete grupos dan una estimación muy incierta.

No elijas `train_test_split` por costumbre. Define primero el uso, identifica observaciones que dependen entre sí (país, empresa, fecha) y mantén esos grupos del mismo lado del corte.

Para evitar mezclar un mismo país entre Train y Test, asignaremos países completos con una semilla reproducible y verificaremos la separación. Con solo siete países, la estimación tendrá mucha incertidumbre y no debe venderse como evaluación definitiva.

> **Nota para quien viene de estadística:** la línea `assert set(g_train).isdisjoint(...)` no es parte del análisis estadístico — es un chequeo de seguridad que detiene el notebook si Train y Test llegaran a compartir algún país, algo que no debería pasar nunca con esta partición. Si nunca se dispara, es una buena señal, no un paso decorativo. El notebook complementario [Día 0.1, Sección 6](00_1_Dia0_Puente_a_Scikit_Learn.ipynb) explica `assert` y cómo leer un error más largo si llegara a fallar.

In [ ]:
X = df[features]
y = df[target]
paises = np.array(sorted(df.iso3.unique()))
paises = np.random.default_rng(42).permutation(paises)
n_test = max(2, round(.25 * len(paises)))
paises_test = set(paises[:n_test])
mask_test = df.iso3.isin(paises_test)
X_train, y_train = X.loc[~mask_test], y.loc[~mask_test]
X_test, y_test = X.loc[mask_test], y.loc[mask_test]
g_train = df.loc[~mask_test, "iso3"]
print("Países Train:", sorted(set(g_train)))
print("Países Test:", sorted(paises_test))
assert set(g_train).isdisjoint(set(df.loc[mask_test, "iso3"]))
print("Tamaños:", X_train.shape, X_test.shape)


### Comprueba la partición antes de entrenar

La celda baraja los códigos de país con semilla fija y deja dos países completos en Test; filas restantes de los países seleccionados no entran en Train. `assert` falla si accidentalmente hay un código compartido. Debes informar qué países quedaron en cada lado porque siete grupos son pocos y el resultado depende fuertemente de esta elección.

Un test de dos países no representa con precisión toda América Latina; sirve para enseñar prevención de fuga entre años correlacionados.

### Comprueba tamaño y composición antes del ajuste

Mira los códigos impresos y las dimensiones. Verifica que ningún país aparezca en ambos subconjuntos y cuántas observaciones completas aporta cada uno. Argentina tiene menos filas completas por faltantes de inflación; por tanto, una selección aleatoria de dos países puede producir un test pequeño. El número de filas no reemplaza el número de unidades independientes.

## 3. Baseline y búsqueda de hiperparámetros

El baseline predice la media de crecimiento del Train. El modelo será Random Forest Regressor; comparamos profundidad y tamaño mínimo de hoja. GroupKFold mantiene países completos separados dentro de cada fold. El escalado no es necesario para árboles, pero el Pipeline mantiene el diseño listo para agregar transformaciones sin leakage.

## Validación cruzada agrupada

`GroupKFold` divide grupos (aquí, países), no filas individuales. En cada fold entrena con unos países y valida con otros, reutilizando los datos de desarrollo varias veces. El promedio de MAE estima el error típico de esas divisiones; la desviación informa cuánto varía.

Con pocos países, no hay muchos folds realmente distintos. La desviación es importante y la media puede cambiar mucho si un país extremo se mueve de fold. CV no elimina sesgo del dataset ni reemplaza una validación temporal o externa.

### Qué calcula cada fold

En cada repetición, `GroupKFold` retiene países completos para validación, ajusta el modelo con los demás y calcula MAE en los retenidos. Al terminar, cada grupo ha servido para validar una vez. El promedio resume esos ejercicios; la desviación ayuda a ver cuánto depende de qué países se retuvieron.

Si una configuración gana por una diferencia mínima frente a su variabilidad, descríbela como empate práctico. La búsqueda no convierte siete países en miles de casos independientes.

> **Nota para quien viene de estadística:** `param_grid={"model__max_depth": [...], "model__min_samples_leaf": [...]}` usa la convención `paso__parametro` para apuntar a hiperparámetros de un paso específico dentro del `Pipeline` (aquí, el paso llamado `"model"`). `GridSearchCV` prueba todas las combinaciones y valida cada una con `GroupKFold`. El `scoring="neg_mean_absolute_error"` vuelve a ser la convención del signo negativo. Todo esto se construye paso a paso en el notebook complementario [Día 0.1, Secciones 4.4 y 5](00_1_Dia0_Puente_a_Scikit_Learn.ipynb).

In [ ]:
cv = GroupKFold(n_splits=min(3, g_train.nunique()))
baseline = DummyRegressor(strategy="mean")
scores_baseline = cross_val_score(baseline, X_train, y_train, groups=g_train, cv=cv,
                                  scoring="neg_mean_absolute_error", n_jobs=1)
mae_baseline_cv = -scores_baseline.mean()
print("MAE CV baseline, puntos porcentuales:", round(mae_baseline_cv, 3))

pipe = Pipeline([("model", RandomForestRegressor(n_estimators=150, random_state=42))])
busqueda = GridSearchCV(pipe,
    param_grid={"model__max_depth":[2,4,None], "model__min_samples_leaf":[2,5,10]},
    scoring="neg_mean_absolute_error", cv=cv, refit=True, n_jobs=1,
    return_train_score=True)
busqueda.fit(X_train, y_train, groups=g_train)
print("Mejores parámetros:", busqueda.best_params_)
print("MAE CV medio:", round(-busqueda.best_score_, 3))


### Baseline y unidades de evaluación

`DummyRegressor(strategy='mean')` aprende en cada Train la media de crecimiento y la usa para sus filas de validación. Su MAE está en **puntos porcentuales de crecimiento del PIB**, no dólares ni puntos de Accuracy. Si el modelo optimizado no mejora frente a esta regla sencilla, no hay razón para preferir su complejidad con estos datos.

Comparamos el baseline con el MAE promedio de GroupKFold sobre los mismos grupos; luego el test final se mide una sola vez.

### Métrica y objetivo antes de probar hiperparámetros

La cuadrícula optimiza MAE, una elección coherente cuando se penaliza linealmente el tamaño del error en puntos de crecimiento. Si un error extremo cuesta mucho más, RMSE o una pérdida asimétrica podría ser más representativa. Cambiar la métrica cambia qué configuración resulta «mejor».

Escribe la unidad y sentido de la métrica antes de abrir `best_params_`; el algoritmo no conoce el costo operativo.

### Interpreta la búsqueda

`best_score_` es negativo porque scikit-learn maximiza scores; cambiamos el signo para leer MAE. Compara MAE de CV con el baseline y `cv_results_` (media y desviación por configuración). Una diferencia pequeña frente a la variabilidad no justifica complejidad adicional. GroupKFold con pocos países genera folds distintos y resultados frágiles; reporta cantidad de grupos y no generalices a todo Ecuador.

## Cómo funciona GridSearchCV aquí

Hay 3 valores para `max_depth` y 3 para `min_samples_leaf`: **9 configuraciones**. Con 3 folds se realizan 27 ajustes de validación, además del reajuste final de la mejor configuración sobre Train (`refit=True`). Cada Random Forest construye 150 árboles. Este tamaño deliberadamente pequeño limita el tiempo de la sesión y hace auditable la cuadrícula.

La biblioteca maximiza un score; para errores utiliza `neg_mean_absolute_error`, por eso `best_score_` es negativo. Cambiamos el signo para volver a MAE: menor es mejor.

### Qué significa ese costo en la práctica

La cuadrícula del ejemplo ya cuantifica sus ajustes en la sección anterior. Lo importante aquí es que probar más candidatos **no crea observaciones independientes**: con pocos países, algunos resultados pueden depender mucho de qué país quede fuera en cada fold. Revisa la desviación de los scores y no declares una mejora sustantiva por una diferencia mínima. La CV pertenece únicamente a Train; Test responde al final si la decisión cerrada se sostiene.


> **Nota para quien viene de estadística:** la secuencia `pd.DataFrame(busqueda.cv_results_)[[...]].copy()` y las líneas con `.pop(...)` encadenan varios métodos uno detrás de otro, aplicándose cada uno sobre el resultado del anterior (ver el notebook complementario [Día 0.1, Sección 9.3](00_1_Dia0_Puente_a_Scikit_Learn.ipynb) con un ejemplo chico). `cv_results_` es, además, uno de esos atributos con guion bajo al final que solo existen después de `.fit()` (Día 0.1, Sección 5.4).

In [ ]:
cv_tabla = pd.DataFrame(busqueda.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf",
    "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]].copy()
cv_tabla["MAE_train"] = -cv_tabla.pop("mean_train_score")
cv_tabla["MAE_CV"] = -cv_tabla.pop("mean_test_score")
cv_tabla = cv_tabla.sort_values("rank_test_score")
cv_tabla.round(3)


### Cómo estudiar `cv_results_`

`MAE_train` describe ajuste en los folds de entrenamiento; `MAE_CV` en sus países retenidos. Si Train es mucho menor que CV, la complejidad podría estar sobreajustando. `std_test_score` (aquí el score tiene signo negativo antes de conversión) refleja variabilidad entre folds; si las diferencias entre candidatos son menores que esa variación, no proclames un ganador claro.

`rank_test_score` ordena el score promedio, no calidad institucional ni estabilidad causal. Anota qué combinación se escogió y por qué.

### Brecha entre Train y CV

Si `MAE_train` es mucho menor que `MAE_CV`, la complejidad podría estar ajustando detalles que no se repiten entre países. Si ambos errores son altos, puede haber subajuste, features insuficientes o mucho ruido. Si CV mejora poco frente al baseline, la complejidad añadida quizá no aporta.

Compara la diferencia con la desviación entre folds: si la mejora es menor que la variación, no proclames un ganador estable.

## 4. Evaluación de Test una sola vez

La configuración se seleccionó exclusivamente mediante CV sobre Train. Ahora medimos Test por países no vistos. No regresamos a cambiar hiperparámetros tras ver el resultado. Si lo hacemos, necesitamos un nuevo conjunto independiente.

## Test es la confirmación, no el selector

La regla del notebook compara baseline y mejor búsqueda en CV; escoge el de menor MAE promedio. Solo después predice los países reservados. Al mostrar Test no hay otra ronda de ajustes. Si Test sugiere una modificación, podemos diseñar otra evaluación o conseguir nuevos datos; no lo reutilizamos como Validation sin reconocer el sesgo.

In [ ]:
if -busqueda.best_score_ < mae_baseline_cv:
    modelo = busqueda.best_estimator_
    nombre_modelo = "Random Forest"
else:
    modelo = DummyRegressor(strategy="mean").fit(X_train, y_train)
    nombre_modelo = "Baseline media"
pred = modelo.predict(X_test)
pd.DataFrame([{"modelo":nombre_modelo, "MAE":mean_absolute_error(y_test,pred), "R2":r2_score(y_test,pred)}]).set_index("modelo").round(3)


### Interpretación ejecutiva

MAE está en puntos porcentuales. Compara RF contra baseline: si no lo supera de forma clara, la conclusión correcta es que no hay evidencia de mejora con esta muestra y diseño. $R^2$ negativo en Test puede ocurrir, especialmente al dejar países completos fuera. Un buen resultado tampoco demuestra causalidad ni garantiza transferibilidad al BCE.

### Qué puede significar un $R^2$ negativo

$R^2$ compara error con la variabilidad del conjunto evaluado. Un valor negativo indica que las predicciones fueron peores que usar la media real de ese conjunto como referencia matemática; esa media de Test no estaría disponible al pronosticar, por eso mantenemos el baseline de Train por separado. Con países no vistos y una muestra pequeña, un resultado negativo no es raro.

La decisión principal se basa en MAE y su unidad; $R^2$ agrega perspectiva, no reemplaza conocer el costo del error.

## 5. Búsqueda aleatoria, visualización y elección de datos

`GridSearchCV` explora todas las combinaciones indicadas; una cuadrícula amplia cuesta mucho. `RandomizedSearchCV` muestrea una cantidad fijada y es útil para espacios grandes. En esta práctica de 3 horas, la cuadrícula pequeña permite inspeccionar cada resultado y evita distraerse con una búsqueda costosa. Para el proyecto final prioriza: fuente descargada y documentada, unidad/target claros, tamaño suficiente, ausencia de datos confidenciales y baseline.

## Grid Search o Random Search

La cuadrícula evalúa todas las combinaciones predefinidas; es transparente cuando son pocas. `RandomizedSearchCV` toma una cantidad finita de muestras de distribuciones o listas y resulta conveniente cuando hay muchas dimensiones de hiperparámetros. Ningún método crea información: la búsqueda puede seleccionar ruido si los datos son escasos.

Para esta clase preferimos nueve combinaciones para poder leerlas. En una aplicación se define primero un rango sustantivamente razonable, un límite de cómputo y una validación acorde al uso.

### Evita sobreajustar también la búsqueda

Si probamos decenas de configuraciones sobre los mismos folds, alguna puede ganar por azar. Cuanto más amplia la búsqueda, más importancia tienen una CV robusta, una regla de selección previa y un test final no consultado.

No ajustes `max_depth` otra vez porque el resultado Test no te gusta. Ese acto convierte Test en otra Validation y vuelve optimista el reporte.

## Proyecto integrador — Tarea 3: modelar y decidir con Validation (20 puntos)

**Práctica guiada, entrega al terminar el Día 6; continúa las Tareas 1 y 2 con el mismo problema.** Regresión: compara una referencia simple (media o persistencia temporal) con al menos un modelo usando MAE/RMSE y una Validation adecuada. Clasificación: contrasta la clase mayoritaria con al menos dos clasificadores y presenta matriz de confusión, Precision, Recall y F1 de la clase positiva además de Accuracy. Clustering: compara varios `k` y/o K-Means–Ward con tamaños, Silhouette, perfiles y estabilidad; no uses Accuracy sin etiquetas reales. En todos los casos muestra una tabla de alternativas sobre **las mismas observaciones de Validation**, explica por qué la métrica responde a tu decisión, registra una limitación y deja **por escrito el procedimiento/modelo seleccionado antes de abrir Test**. La cuadrícula de hiperparámetros y PCA de este notebook son ejemplos: úsalos en tu proyecto solo si ayudan a su pregunta.

**Evidencia:** notebook que extiende las dos tareas anteriores con referencia, modelos, comparación y decisión congelada; Test aún reservado para la práctica final. No existe una cuarta nota de 20 puntos por el reto de optimización del ejemplo. Rúbrica de la Tarea 3 en [`INSTRUMENTOS_EVALUACION_MODULO3.md`](../INSTRUMENTOS_EVALUACION_MODULO3.md).

### Ficha para decidir si un dataset sirve al proyecto

Antes de modelar con el CSV local elegido en el Día 4, revisa la ficha de las Tareas 1–2: fuente, tipo de dato público/sintético, fecha de corte; qué representa cada fila; quién es la población; target o patrón sin etiqueta; cuándo se necesita la respuesta; qué variables estarán disponibles; faltantes y revisiones; baseline o criterio de estabilidad; split y costo de errores.

Para preguntas de trabajo del BCE, prioriza estadísticas públicas autorizadas y documentadas. No subas microdatos institucionales a servicios externos ni elijas un dataset solo por ser grande. Los archivos del repositorio son locales y el mensual es sintético; documenta cualquier alternativa.

### Criterios para aprobar un dataset económico

Busca fuente estable y licencia, fecha de corte, qué representa cada fila, quién es la población, target o patrón, momento de predicción y disponibilidad de variables. Revisa faltantes, revisiones y baseline.

Para un caso del BCE, no mezcles una cifra anual revisada con una variable mensual sin alinear frecuencias. No uses categorías sensibles ni microdatos sin autorización. El panel de este ejercicio es público y reducido; el mensual del Día 7 es sintético.

### Preguntas de cierre

¿Por qué se agrupan países al separar datos? ¿Qué evita CV frente a mirar una sola partición? ¿Por qué una búsqueda de muchos hiperparámetros puede sobreajustar? ¿Qué significa superar al baseline? ¿Por qué la figura de PCA de toda la muestra no es parte del pipeline predictivo?

### Traduce una métrica a una conclusión

Redacta tres piezas: (1) **estimación**: «MAE del baseline/RF en países retenidos fue ___ puntos»; (2) **comparación**: «CV favoreció ___ por ___, con dispersión ___»; (3) **límite**: «solo hay siete países, y el test contiene ___; por eso no generalizo a ___». Esta forma evita convertir un score pequeño en una promesa institucional.

El Día 7 retoma el procedimiento que escogiste **sin mirar Test**. La separación temporal aplica a preguntas futuras; clasificación de clientes o perfiles país-año pueden requerir otras particiones justificadas. Entonces se realiza la comprobación final y la comunicación ejecutiva.

### Cierra con una ficha de decisión

Escribe: pregunta, unidad de observación, target, momento de predicción, baseline, métrica/costo, split, hiperparámetros probados, criterio de selección y limitación principal. La ficha obliga a explicar para qué sirve el análisis antes de nombrar el algoritmo.

El Día 7 reutiliza esta ficha en el proyecto y añade automatización y un caso de forecasting.

## Cierre de la Tarea 3 (20 puntos) antes de la práctica final

Trae el **mismo notebook** desarrollado en las Tareas 1 (20) y 2 (20), no comiences otro dataset. Al terminar, enseña (1) pregunta, fuente y unidad; (2) target o criterio exploratorio; (3) baseline o referencia; (4) partición acorde al uso —países no vistos, periodos futuros y filas intercambiables son **preguntas diferentes**—; (5) resultados de alternativas sobre la misma Validation/CV; (6) **decisión congelada** para la Tarea Final; (7) dos límites. Si elegiste clustering, sustituye métricas supervisadas por tamaños, perfiles, Silhouette y estabilidad; usa el Test retenido solo para comprobación final de perfiles, no para inventar etiquetas.

**Profundización autónoma:** completa resultados y una síntesis de una página antes del Día 7. No optimices mirando Test. La **conclusión escrita dentro del notebook**, sin exposición oral, debe explicar por qué el protocolo escogido responde a la pregunta, no solo mostrar que ejecutó una búsqueda.
